# Diabetes Prediction — Sadman's ADASYN Model v1

**BRFSS 2015 Diabetes Health Indicators Dataset**

## Objective

This notebook evaluates diabetes prediction using **ADASYN** as the
training-data resampling technique.

The team's original preprocessing pipeline uses:

    Raw BRFSS data
            ↓
    Stratified 80/20 split
            ↓
      ┌─────┴─────┐
      ↓           ↓
    Train       Holdout
      ↓           ↓
 SMOTE-ENN     untouched
      ↓           ↓
   models     evaluation

For Sadman's experiment, we replace SMOTE-ENN with **ADASYN**:

    Raw BRFSS data
            ↓
    Same stratified 80/20 split
            ↓
      ┌─────┴─────┐
      ↓           ↓
    Train       Holdout
      ↓           ↓
    ADASYN      untouched
      ↓           ↓
   models     evaluation

The purpose is to compare the effect of different class-imbalance
techniques while keeping the dataset split, feature groups, models,
and evaluation procedure as consistent as possible.

## Important

The 20% holdout is never resampled.

It is kept as real, untouched BRFSS data and is used for the headline
evaluation metrics.


## 1. Project setup

This cell locates the repository root so that the notebook can import
the existing project code and access the configuration file.

We use the project's existing configuration rather than hardcoding
experimental settings wherever possible.


In [ ]:
import sys
from pathlib import Path

# Find the repository root.
# This allows the notebook to work even if Jupyter was started from
# notebooks/sadman/ rather than the repository root.

REPO_ROOT = Path.cwd()

while (
    not (REPO_ROOT / "configs").is_dir()
    and REPO_ROOT != REPO_ROOT.parent
):
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)

## 2. Imports and configuration

The project configuration contains the random seed, target-column name,
test-set size, W&B project information, and artifact names.

The project's configured random seed is used throughout the experiment
for reproducibility.

The original preprocessing code also uses this seed when creating the
stratified train/holdout split. The project uses `set_seed()` from
`src/utils/seed.py` for the global random-number generators.


In [ ]:
import pandas as pd
import numpy as np
import yaml
import wandb

# Load project configuration
CONFIG_PATH = REPO_ROOT / "configs" / "resample_smoteenn.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Read important configuration values
TARGET = cfg["data"]["target_column"]
SEED = int(cfg["random_seed"])
TEST_SIZE = float(cfg["data"]["test_size"])

OWNER = "sadman"

print("Target:", TARGET)
print("Random seed:", SEED)
print("Test size:", TEST_SIZE)
print("W&B project:", cfg["wandb"]["project"])

## 3. Set the project's random seed

We use the existing `set_seed()` function from `src/utils/seed.py`.

We do not create a new seed implementation in this notebook.

The project's helper seeds Python's `random` module, NumPy's global RNG,
and `PYTHONHASHSEED`.

The same seed is also passed explicitly to models and other algorithms
that support `random_state`.

This makes the experiment reproducible and follows the team's existing
convention.

In [ ]:
from src.utils.seed import set_seed

# Use the seed defined by the project configuration
set_seed(SEED)

print(f"Global seed set to {SEED}")


## 4. Load the raw BRFSS dataset

Unlike Mahdi's modelling notebook, we cannot use the
`brfss-smoteenn-resampled` artifact because that dataset has already
been processed with SMOTE-ENN.

For the ADASYN experiment, we need the training data **before**
SMOTE-ENN.

Therefore, we load the original raw BRFSS CSV.

We will reproduce the project's exact train/holdout split in the next
section.

The raw dataset should contain approximately:

- 253,680 rows
- 22 columns
- 21 input features
- 1 binary target (`diabetes_binary`)

In [ ]:
RAW_PATH = (
    REPO_ROOT
    / "data"
    / "raw"
    / "diabetes_binary_health_indicators_BRFSS2015.csv"
)

df = pd.read_csv(RAW_PATH)

# The project's loader normalizes column names to lowercase.
df.columns = df.columns.str.lower()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 5. Basic dataset validation

Before performing any experiment, we verify that the raw dataset is
the expected BRFSS binary diabetes dataset.

We check:

1. The expected target column exists.
2. There are no missing values.
3. The target is binary.
4. The dataset has the expected number of rows and columns.

This is a sanity check and prevents accidentally running the experiment
on the wrong Kaggle file.

In [ ]:
assert TARGET in df.columns, (
    f"Target column '{TARGET}' was not found."
)

print("Target values:")
print(df[TARGET].value_counts().sort_index())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nDataset shape:")
print(df.shape)

## 6. Reproduce the project's original 80/20 split

### Why are we splitting here?

Normally, Mahdi's modelling notebook does NOT perform a train/test split.

That is correct.

The team's preprocessing pipeline performs the split first:

    raw dataset
         ↓
    stratified train/holdout split
         ↓
    80% training
         ↓
    SMOTE-ENN

Mahdi then receives the resulting artifacts from W&B.

I do not have access to the original pre-SMOTE-ENN training artifact,
so I must reconstruct the **output of that preprocessing stage** locally.

This is not a new experimental split.

It is an exact reproduction of the split already defined by the
project's preprocessing code.

The project uses:

    test_size = 0.20
    random_state = 42
    stratify = diabetes_binary

Therefore, we reproduce those exact settings here.

The original preprocessing implementation performs this split before
any resampling is applied. 

In [ ]:
from sklearn.model_selection import train_test_split

# IMPORTANT:
# This is the same split used by the project's preprocessing pipeline.
#
# We are NOT splitting the already-resampled data.
# We are splitting the original raw dataset BEFORE ADASYN.

train_df, holdout_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[TARGET],
)

# Match the project's preprocessing implementation
train_df = train_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print("Training shape :", train_df.shape)
print("Holdout shape  :", holdout_df.shape)

## 7. Verify the recreated holdout against W&B

This is an important validation step.

The project already has an official W&B artifact:

    brfss-holdout-test

That artifact contains the untouched 20% holdout produced by the
preprocessing pipeline.

Because we recreated the exact split using the same:

- raw dataset
- seed
- test size
- stratification

our locally recreated `holdout_df` should contain exactly the same rows.

We therefore compare our holdout against the official W&B artifact.

If they are identical, we have confirmed that our local reconstruction
matches the team's original preprocessing split.

In [ ]:
# Start a W&B run for this experiment
run = wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"]["entity"],
    job_type="train",
    name=f"{OWNER}-train-adasyn-v1",
    tags=[OWNER, "adasyn", "modelling"],
)

print("W&B run initialized.")

In [ ]:
# Download the official untouched holdout artifact.
# We use an explicit version when possible so that the evaluation set
# cannot silently change during the experiment.

HOLDOUT_ARTIFACT_NAME = cfg["holdout_artifact"]["name"]

holdout_artifact = run.use_artifact(
    f"{HOLDOUT_ARTIFACT_NAME}:v0"
)

holdout_dir = Path(holdout_artifact.download())

official_holdout = pd.read_csv(
    next(holdout_dir.glob("*.csv"))
)

official_holdout.columns = official_holdout.columns.str.lower()

print("Official W&B holdout shape:", official_holdout.shape)

In [ ]:
# Compare the two datasets after sorting.
# Sorting is necessary because train_test_split preserves the selected
# rows but the row ordering may differ between files.

my_holdout = (
    holdout_df
    .sort_values(list(holdout_df.columns))
    .reset_index(drop=True)
)

wb_holdout = (
    official_holdout
    .sort_values(list(official_holdout.columns))
    .reset_index(drop=True)
)

holdout_identical = my_holdout.equals(wb_holdout)

print("Local holdout shape :", my_holdout.shape)
print("W&B holdout shape   :", wb_holdout.shape)
print("\nIdentical:", holdout_identical)

## 8. Build training and test features

At this point:

- `train_df` = the original 80% training data
- `holdout_df` = the original 20% holdout
- `official_holdout` = the team's W&B evaluation set

The holdout remains completely untouched.

We now separate features (`X`) from the binary diabetes target (`y`).

No resampling has happened yet.


In [ ]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].astype(int)

# Use the official W&B holdout for headline evaluation.
X_test = official_holdout.drop(columns=[TARGET])
y_test = official_holdout[TARGET].astype(int)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")

## 9. Check the class imbalance before ADASYN

The original BRFSS binary diabetes dataset is strongly imbalanced.

Class 0 represents people without the target condition, while class 1
represents people with the target condition.

ADASYN is being used because standard training on the raw distribution
can make a classifier favor the majority class.

Before applying ADASYN, we record the original class distribution.

The test set is also checked separately because it must remain
representative of the original data distribution.

In [ ]:
print("Training class distribution BEFORE ADASYN:")
print(y_train.value_counts().sort_index())

print("\nTraining proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nHoldout class distribution:")
print(y_test.value_counts().sort_index())

print("\nHoldout proportions:")
print(y_test.value_counts(normalize=True).sort_index())

## 10. Apply ADASYN to the training data only

### What is ADASYN?

ADASYN stands for **Adaptive Synthetic Sampling**.

It is an oversampling technique designed to address class imbalance.

Instead of generating the same number of synthetic minority examples
everywhere, ADASYN focuses more strongly on minority-class examples
that are difficult to learn.

Conceptually:

    Minority samples
          ↓
    Examine local neighbourhood
          ↓
    Identify difficult minority examples
          ↓
    Generate more synthetic samples
    around difficult regions
          ↓
    More balanced training set

This experiment asks whether ADASYN produces better diabetes-prediction
performance than the team's SMOTE-ENN preprocessing approach.

### Important data-leakage rule

ADASYN is fitted ONLY on:

    X_train / y_train

It is NEVER applied to:

    X_test / y_test

The holdout therefore remains real, untouched data.

In [ ]:
from imblearn.over_sampling import ADASYN

# Create ADASYN using the same project seed.
adasyn = ADASYN(
    random_state=SEED
)

# IMPORTANT:
# Only the training set is resampled.
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(
    X_train,
    y_train
)

# Convert back to DataFrame/Series so feature names are preserved.
X_train_adasyn = pd.DataFrame(
    X_train_adasyn,
    columns=X_train.columns
)

y_train_adasyn = pd.Series(
    y_train_adasyn,
    name=TARGET
).astype(int)

print("ADASYN completed.")

In [ ]:
print("Class distribution BEFORE ADASYN:")
print(y_train.value_counts().sort_index())

print("\nClass distribution AFTER ADASYN:")
print(y_train_adasyn.value_counts().sort_index())

print("\nShape BEFORE ADASYN:")
print(X_train.shape)

print("\nShape AFTER ADASYN:")
print(X_train_adasyn.shape)

## 11. Verify that the holdout was not changed

This is a final leakage check before modelling.

ADASYN should only have modified the training data.

The holdout must remain exactly the same.

We therefore confirm that:

- `X_test` still contains the original holdout features.
- `y_test` still contains the original holdout target.
- No synthetic examples have been introduced into the test set.

In [ ]:
assert X_test.shape[0] == official_holdout.shape[0]
assert y_test.shape[0] == official_holdout.shape[0]

print("Holdout remains untouched.")
print("Holdout rows:", len(X_test))

# 12. Modelling setup

From this point onward, the modelling procedure follows Mahdi's
notebook as closely as possible.

We keep the same:

- five model types
- model random seed
- feature groups
- feature-selection methods
- evaluation metrics
- untouched holdout evaluation

The only intended methodological change is:

    Mahdi: SMOTE-ENN training data

    Sadman: ADASYN training data

This is important because otherwise differences in the results could be
caused by changing multiple parts of the experiment simultaneously.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt

from sklearn.feature_selection import (
    mutual_info_classif,
    RFE
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 13. Model factory

We create fresh model instances for every feature group.

This is important because a fitted model should never carry state from
one feature group into another.

The models are kept the same as the team's existing modelling notebook:

1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM
5. K-Nearest Neighbours

The same random seed is passed to models that support `random_state`.

In [ ]:
MODEL_ORDER = [
    "logreg",
    "random_forest",
    "xgboost",
    "lightgbm",
    "knn",
]


def make_models(seed):
    """
    Create fresh, unfitted model instances.

    The model configurations follow Mahdi's modelling notebook.
    """

    return {
        "logreg": Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000,
                    random_state=seed,
                    n_jobs=-1
                )
            ),
        ]),

        "random_forest": RandomForestClassifier(
            random_state=seed,
            n_jobs=-1
        ),

        "xgboost": XGBClassifier(
            random_state=seed,
            n_jobs=-1,
            eval_metric="logloss"
        ),

        "lightgbm": LGBMClassifier(
            random_state=seed,
            n_jobs=-1,
            verbose=-1
        ),

        "knn": Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                KNeighborsClassifier(
                    n_jobs=-1
                )
            ),
        ]),
    }


print("Models:", MODEL_ORDER)

## 14. Evaluation metrics

Because the original test set is imbalanced, accuracy alone is not a
sufficient measure of model performance.

We therefore calculate:

### Precision

Of the people predicted as diabetic, how many are actually positive?

### Recall

Of the actual positive cases, how many did the model identify?

### F1-score

Harmonic mean of precision and recall.

### ROC-AUC

Measures ranking/discrimination ability across classification
thresholds.

### PR-AUC

Measures the precision-recall tradeoff and is particularly useful when
the positive class is relatively uncommon.

### Balanced accuracy

Average of sensitivity for the positive and negative classes.

We use the same metrics as Mahdi so that the two resampling approaches
can be compared consistently.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """
    Compute the same evaluation metrics used in the team's
    modelling notebook.
    """

    return {
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_true,
            y_proba
        ),

        "pr_auc": average_precision_score(
            y_true,
            y_proba
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
    }

# 15. Define the feature groups

We now divide the predictors into the same domain-based groups used
by the team's modelling experiment.

The groups are:

- Biological
- Socioeconomic
- Lifestyle
- Combined all features

We also later create two hybrid groups:

- Hybrid MI: top 10 features selected by Mutual Information
- Hybrid RFE: top 10 features selected using Recursive Feature Elimination

The important point is that the feature-group definitions remain the
same as Mahdi's experiment.

We are changing the resampling method, not the research question about
feature groups.

In [ ]:
def build_domain_groups(exclude=()):
    """
    Build the same domain feature groups used in the team's
    modelling experiment.

    `exclude` is used later for the sensitivity analysis.
    """

    exclude = set(exclude)

    groups = {
        "biological": [
            c for c in [
                "bmi",
                "highchol",
                "cholcheck",
                "highbp",
                "heartdiseaseorattack",
                "stroke",
                "age",
                "sex",
            ]
            if c not in exclude
        ],

        "socioeconomic": [
            c for c in [
                "income",
                "education",
                "anyhealthcare",
                "nodocbccost",
            ]
            if c not in exclude
        ],

        "lifestyle": [
            c for c in [
                "smoker",
                "physactivity",
                "fruits",
                "veggies",
                "hvyalcoholconsump",
                "menthlth",
                "physhlth",
                "diffwalk",
                "genhlth",
            ]
            if c not in exclude
        ],
    }

    # All available features, excluding any requested columns.
    groups["combined_all"] = [
        c for c in X_train.columns
        if c not in exclude
    ]

    return groups

## 16. Mutual Information and RFE

The project also creates hybrid feature groups.

### Mutual Information

Mutual Information estimates how much information each feature
provides about the target.

A larger value indicates a stronger statistical dependency between
the feature and the target.

We rank all candidate features and select the top 10.

### RFE

Recursive Feature Elimination repeatedly fits a model and removes
less important features until the desired number remains.

Here we use Logistic Regression with scaling and select 10 features.

Both methods are fitted using the training data only.

The untouched test set is never used for feature selection.

In [ ]:
def compute_hybrid_top10(pool_cols, seed):
    """
    Select the top 10 features using:

    1. Mutual Information
    2. RFE with scaled Logistic Regression

    Feature selection is performed using training data only.
    """

    X_pool = X_train_adasyn[pool_cols]

    # -------------------------
    # Mutual Information
    # -------------------------

    mi_scores = mutual_info_classif(
        X_pool,
        y_train_adasyn,
        random_state=seed
    )

    mi_series = (
        pd.Series(
            mi_scores,
            index=pool_cols
        )
        .sort_values(ascending=False)
    )

    top10_mi = list(
        mi_series.index[:10]
    )

    # -------------------------
    # RFE
    # -------------------------

    rfe_pipe = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "rfe",
            RFE(
                LogisticRegression(
                    max_iter=1000,
                    random_state=seed
                ),
                n_features_to_select=10
            )
        ),
    ])

    rfe_pipe.fit(
        X_pool,
        y_train_adasyn
    )

    support = (
        rfe_pipe
        .named_steps["rfe"]
        .support_
    )

    top10_rfe = [
        str(c)
        for c in np.array(pool_cols)[support]
    ]

    return (
        mi_series,
        top10_mi,
        top10_rfe
    )

In [ ]:
# Compute hybrid feature groups using the ADASYN training set.

mi_series_full, top10_mi, top10_rfe = compute_hybrid_top10(
    list(X_train_adasyn.columns),
    SEED
)

print("MI top 10:")
print(top10_mi)

print("\nRFE top 10:")
print(top10_rfe)

print("\nOverlap:")
print(
    sorted(
        set(top10_mi) & set(top10_rfe)
    )
)

In [ ]:
# Build all feature groups.

domain_groups = build_domain_groups()

domain_groups["hybrid_mi"] = top10_mi
domain_groups["hybrid_rfe"] = top10_rfe

GROUP_ORDER = list(domain_groups)

print("Feature groups:")
for name, cols in domain_groups.items():
    print(f"{name:20s}: {len(cols)} features")

# 17. Main experiment

Now we perform the central experiment.

For every feature group:

1. Select only that group's training features.
2. Select the same columns from the holdout set.
3. Create fresh instances of all five models.
4. Train each model on the ADASYN-resampled training data.
5. Predict on the untouched holdout.
6. Calculate the six evaluation metrics.
7. Save the confusion matrix.

The critical distinction is:

    TRAIN:
        X_train_adasyn
        y_train_adasyn

    TEST:
        X_test
        y_test

The test set is never used during fitting.

In [ ]:
def run_experiment(groups, seed):
    """
    Run every model on every feature group.

    Training uses ADASYN-resampled data.
    Evaluation uses the untouched holdout.

    Returns:
        results_df
        confusion_matrices
    """

    rows = []
    cms = {}

    for group_name in groups:

        cols = groups[group_name]

        # Select the same feature columns from train and test.
        X_tr = X_train_adasyn[cols]
        X_te = X_test[cols]

        # Fresh models for this feature group.
        models = make_models(seed)

        for model_name in MODEL_ORDER:

            model = models[model_name]

            # -------------------------
            # TRAIN
            # -------------------------

            model.fit(
                X_tr,
                y_train_adasyn
            )

            # -------------------------
            # TEST
            # -------------------------

            y_pred = model.predict(X_te)

            y_proba = model.predict_proba(X_te)[:, 1]

            # -------------------------
            # METRICS
            # -------------------------

            cms[
                (group_name, model_name)
            ] = confusion_matrix(
                y_test,
                y_pred
            )

            rows.append({
                "group": group_name,
                "model": model_name,
                **compute_metrics(
                    y_test,
                    y_pred,
                    y_proba
                ),
            })

    return pd.DataFrame(rows), cms

# 18. Run the main experiment

This will train:

    6 feature groups × 5 models

for a total of:

    30 model experiments

Each experiment is evaluated against the same untouched holdout.

The main result table will therefore contain one row for every:

    feature group × model

combination.

In [ ]:
results_df, cms_main = run_experiment(
    domain_groups,
    SEED
)

# Save locally using an ADASYN-specific filename.
results_df.to_csv(
    "results_group_models_adasyn.csv",
    index=False
)

print(
    results_df
    .sort_values("pr_auc", ascending=False)
    .to_string(index=False)
)

## 19. Why PR-AUC is especially important here

The evaluation holdout retains the original imbalanced distribution.

Therefore, a model can potentially obtain a good-looking accuracy
while still performing poorly on the minority diabetes class.

PR-AUC focuses on the precision-recall relationship and is therefore
particularly useful for evaluating performance on the positive class
in this imbalanced setting.

We still report ROC-AUC, precision, recall, F1, and balanced accuracy
so that the results are not dependent on one metric alone.

In [ ]:
print("\nResults sorted by PR-AUC:")
display(
    results_df
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

# 20. Confusion matrices

A confusion matrix shows:

                 Predicted
                 0       1

Actual 0        TN      FP

Actual 1        FN      TP

This lets us see the type of errors each model makes.

In particular, false negatives are important because they represent
positive diabetes cases that the model failed to identify.

In [ ]:
def plot_confusion_grid(
    cms,
    groups_order,
    models_order,
    title
):
    """
    Plot confusion matrices in a compact grid.

    Rows = feature groups
    Columns = models
    """

    n_rows = len(groups_order)
    n_cols = len(models_order)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(
            1.8 * n_cols,
            1.8 * n_rows
        ),
        squeeze=False
    )

    for i, group_name in enumerate(groups_order):

        for j, model_name in enumerate(models_order):

            ax = axes[i, j]

            cm = cms[
                (group_name, model_name)
            ]

            ax.imshow(
                cm,
                cmap="Blues",
                vmin=0
            )

            for (r, c), val in np.ndenumerate(cm):

                ax.text(
                    c,
                    r,
                    str(val),
                    ha="center",
                    va="center",
                    fontsize=7
                )

            ax.set_xticks([])
            ax.set_yticks([])

            if i == 0:
                ax.set_title(
                    model_name,
                    fontsize=8
                )

            if j == 0:
                ax.set_ylabel(
                    group_name,
                    fontsize=8,
                    rotation=0,
                    ha="right",
                    va="center"
                )

    fig.suptitle(
        title,
        fontsize=11
    )

    fig.tight_layout()

    return fig

In [ ]:
fig_cm = plot_confusion_grid(
    cms_main,
    GROUP_ORDER,
    MODEL_ORDER,
    "ADASYN — Confusion Matrices"
)

plt.show()
plt.close(fig_cm)

# 21. Compare ROC-AUC and PR-AUC

We now visualize the two main ranking-based metrics across feature
groups and models.

These plots are for comparing the experimental results and making it
easier to see patterns.

The numerical CSV remains the authoritative result table.

In [ ]:
def plot_grouped_bar(
    df,
    metric,
    title
):
    groups_order = [
        g for g in GROUP_ORDER
        if g in df["group"].unique()
    ]

    x = np.arange(
        len(groups_order)
    )

    n_models = len(MODEL_ORDER)

    width = 0.8 / n_models

    fig, ax = plt.subplots(
        figsize=(9, 4)
    )

    for i, model_name in enumerate(
        MODEL_ORDER
    ):

        vals = [
            df.loc[
                (df["group"] == g)
                &
                (df["model"] == model_name),
                metric
            ].iloc[0]
            for g in groups_order
        ]

        ax.bar(
            x + i * width
            - 0.4
            + width / 2,
            vals,
            width=width,
            label=model_name
        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        groups_order,
        rotation=20,
        ha="right"
    )

    ax.set_ylabel(metric)

    ax.set_title(title)

    ax.legend(
        fontsize=8,
        frameon=False
    )

    fig.tight_layout()

    return fig

In [ ]:
fig_roc = plot_grouped_bar(
    results_df,
    "roc_auc",
    "ADASYN — ROC-AUC by Feature Group and Model"
)

plt.show()
plt.close(fig_roc)

In [ ]:
fig_pr = plot_grouped_bar(
    results_df,
    "pr_auc",
    "ADASYN — PR-AUC by Feature Group and Model"
)

plt.show()
plt.close(fig_pr)

# 22. Sensitivity analysis: remove possible downstream proxy features

The original modelling experiment also performs a sensitivity analysis.

The following features are treated as potentially downstream of an
existing diabetes condition rather than purely pre-diagnosis risk
factors:

- `genhlth`
- `menthlth`
- `physhlth`
- `diffwalk`

The purpose of this sensitivity analysis is not to claim that these
features are invalid.

Instead, we ask:

> Does model performance or the ranking of feature groups change when
> these potentially downstream variables are removed?

We repeat the experiment rather than simply deleting the columns from
the final results.

This means feature selection is also recomputed without these features.

In [ ]:
PROXY_COLS = [
    "genhlth",
    "menthlth",
    "physhlth",
    "diffwalk",
]

print("Proxy features removed in sensitivity analysis:")
print(PROXY_COLS)

In [ ]:
# Build a candidate feature pool without the proxy features.

pool_no_proxy = [
    c
    for c in X_train_adasyn.columns
    if c not in PROXY_COLS
]

print("Original number of features:", len(X_train_adasyn.columns))
print("Features after exclusion:", len(pool_no_proxy))

In [ ]:
# Recalculate Mutual Information and RFE after removing the proxy features.

mi_series_np, top10_mi_np, top10_rfe_np = (
    compute_hybrid_top10(
        pool_no_proxy,
        SEED
    )
)

print("MI top 10 without proxy features:")
print(top10_mi_np)

print("\nRFE top 10 without proxy features:")
print(top10_rfe_np)

In [ ]:
# Rebuild the domain groups with the proxy features removed.

no_proxy_groups = build_domain_groups(
    exclude=PROXY_COLS
)

no_proxy_groups["hybrid_mi"] = top10_mi_np
no_proxy_groups["hybrid_rfe"] = top10_rfe_np

print("Feature groups without proxy features:")

for name, cols in no_proxy_groups.items():
    print(
        f"{name:20s}: {len(cols)} features"
    )

# 23. Run the ADASYN sensitivity experiment

We now repeat the exact same model evaluation using the reduced
feature groups.

The resampling method remains ADASYN.

The holdout remains the same untouched W&B holdout.

The only difference from the main experiment is the removal of the
four proxy features.

In [ ]:
results_no_proxy_df, cms_no_proxy = run_experiment(
    no_proxy_groups,
    SEED
)

results_no_proxy_df.to_csv(
    "results_group_models_adasyn_no_proxy.csv",
    index=False
)

print(
    results_no_proxy_df
    .sort_values("pr_auc", ascending=False)
    .to_string(index=False)
)

# 24. Compare feature-group rankings

We now calculate the average PR-AUC of each feature group across the
five models.

This gives us a simple way to ask:

> Which feature group performs best on average?

We calculate the ranking both before and after removing the possible
downstream proxy features.

If the ranking changes, that suggests the apparent usefulness of the
feature groups may partly depend on those variables.

In [ ]:
def group_ranking(df):
    return (
        df
        .groupby("group")["pr_auc"]
        .mean()
        .reindex(
            GROUP_ORDER
        )
        .sort_values(
            ascending=False
        )
    )


rank_main = group_ranking(
    results_df
)

rank_no_proxy = group_ranking(
    results_no_proxy_df
)

print(
    "Ranking with proxy features:"
)

print(
    list(rank_main.index)
)

print(
    "\nRanking without proxy features:"
)

print(
    list(rank_no_proxy.index)
)

changed = (
    list(rank_main.index)
    !=
    list(rank_no_proxy.index)
)

print(
    f"\nGroup ranking "
    f"{'CHANGED' if changed else 'DID NOT CHANGE'} "
    "after removing proxy features."
)

# 25. Save the ADASYN results to W&B

The experiment results are saved as a separate W&B artifact.

We use a different artifact name from Mahdi's SMOTE-ENN results so
that the two experiments are not confused.

Main results:

    group-model-results-adasyn

Sensitivity results:

    group-model-results-adasyn-no-proxy

This allows the team to compare the two resampling approaches in W&B.

In [ ]:
# Log the main result table to W&B.

run.log({
    "results/adasyn_table":
        wandb.Table(
            dataframe=results_df
        )
})

In [ ]:
# Upload main ADASYN results as a W&B artifact.

results_artifact = wandb.Artifact(
    "group-model-results-adasyn",
    type="results",
    description=(
        "ADASYN results across feature groups and models. "
        "Models are trained on ADASYN-resampled training data "
        "and evaluated on the shared untouched BRFSS holdout."
    )
)

results_artifact.add_file(
    "results_group_models_adasyn.csv"
)

run.log_artifact(
    results_artifact
)

In [ ]:
# Upload sensitivity-analysis results.

no_proxy_artifact = wandb.Artifact(
    "group-model-results-adasyn-no-proxy",
    type="results",
    description=(
        "ADASYN results with genhlth, menthlth, "
        "physhlth, and diffwalk removed as a "
        "sensitivity analysis."
    )
)

no_proxy_artifact.add_file(
    "results_group_models_adasyn_no_proxy.csv"
)

run.log_artifact(
    no_proxy_artifact
)

# 26. Final experiment summary

The experiment performed in this notebook is:

    BRFSS 2015 raw dataset
              ↓
    Same stratified 80/20 split
              ↓
       ┌──────┴──────┐
       ↓             ↓
    80% train      20% holdout
       ↓             ↓
     ADASYN       untouched
       ↓             ↓
    5 models       evaluation
       ↓
    6 feature groups
       ↓
    Precision / Recall / F1
    ROC-AUC / PR-AUC
    Balanced Accuracy

The purpose is to provide the team's **ADASYN modelling results**.

These can later be compared against the corresponding SMOTE-ENN
results using the same:

- raw dataset
- train/holdout split
- feature groups
- models
- evaluation metrics
- random seed

In [ ]:
print("=" * 70)
print("ADASYN EXPERIMENT COMPLETE")
print("=" * 70)

print("\nSeed:", SEED)
print("Training rows before ADASYN:", len(X_train))
print("Training rows after ADASYN :", len(X_train_adasyn))
print("Holdout rows               :", len(X_test))

print("\nBest results by PR-AUC:")
display(
    results_df
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .head(10)
)

In [ ]:
# Finish the W&B run.

run.finish()

print("W&B run finished.")